# 01_exploration: Research Questions & Initial EDA
## Comeback Analytics — Gaussian Process Regression on Shot Quality Trajectories

**Objective:** Define research hypotheses, load MoneyPuck data, and explore shot quality patterns in third-period comeback situations.

**Expected outputs:**
- Cleaned dataset filtered to comeback situations (down by 2, 3rd period)
- EDA plots (raw shot counts, xGoal distributions by time window)
- Summary statistics and data validation checks


## 1. Research Framework

### Research Question
**Do teams trailing by two goals in the third period exhibit increasing shot quality (expected goals) over time as they mount a comeback attempt?**

### Hypotheses

**H1 (Primary):** Shot quality (xGoal per shot) increases monotonically in comeback situations as teams shift from deficit-management to offensive pressure.

**H2 (Secondary):** Shot volume increases along with quality—teams may increase both the quantity and intensity of scoring chances.

**H3 (Mechanism):** The effect is driven by player positioning and shot selection changes, not by lucky/unlucky variation or random noise.

### Key Construct: Comeback Situation
- **Definition:** Third period, trailing by exactly 2 goals (or 3 for robustness checks)
- **Rationale:** Down 2 is "comeback-feasible" (within reach in 20 minutes); down 3+ is less actionable
- **Temporal binning:** Four 5-minute windows (0–5, 5–10, 10–15, 15–20 min of period)
- **Data source:** MoneyPuck 2024–25 season, all teams, 5-on-5 situations

### Statistical Framework
- **Response variable:** xGoal per shot (continuous)
- **Predictor:** Time elapsed in period (continuous, 0–1200 seconds)
- **Smoothness assumption:** Shot quality follows a smooth trajectory (justifies GP regression with Matérn kernel)
- **Uncertainty quantification:** Credible bands around posterior mean


## 2. Data Pipeline Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define paths
DATA_DIR = Path('../data')  # Adjust to your MoneyPuck data location
OUTPUT_DIR = Path('../output/exploration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

### 2.1 Load Raw Data


In [ ]:
# Load shot-level data from MoneyPuck
# Assumes CSV with columns: gameId, period, time, team, xGoal, goal, homeScore, awayScore, etc.

shots_df = pd.read_csv(DATA_DIR / 'shots_2024_25.csv')
print(f"Raw shots dataset shape: {shots_df.shape}")
print(f"Columns: {shots_df.columns.tolist()}")
print(f"\nFirst few rows:")
shots_df.head()

### 2.2 Data Cleaning & Filtering


In [ ]:
# Ensure data types
shots_df['gameId'] = shots_df['gameId'].astype(str)
shots_df['period'] = shots_df['period'].astype(int)
shots_df['time'] = pd.to_numeric(shots_df['time'], errors='coerce')
shots_df['xGoal'] = pd.to_numeric(shots_df['xGoal'], errors='coerce')
shots_df['homeScore'] = pd.to_numeric(shots_df['homeScore'], errors='coerce')
shots_df['awayScore'] = pd.to_numeric(shots_df['awayScore'], errors='coerce')

# Remove rows with missing critical values
initial_count = len(shots_df)
shots_df = shots_df.dropna(subset=['gameId', 'period', 'time', 'xGoal', 'homeScore', 'awayScore'])
print(f"Removed {initial_count - len(shots_df)} rows with missing values")

# Filter to 3rd period only
shots_3p = shots_df[shots_df['period'] == 3].copy()
print(f"\n3rd period shots: {len(shots_3p)}")

# Calculate score differential for each shot
# (score at the time of the shot from the shooting team's perspective)
shots_3p['score_diff'] = shots_3p.apply(
    lambda row: row['homeScore'] - row['awayScore'] if row['team'] == 'home' else row['awayScore'] - row['homeScore'],
    axis=1
)

print(f"\nScore differential distribution (3rd period):")
print(shots_3p['score_diff'].value_counts().sort_index())

### 2.3 Filter to Comeback Situations


In [ ]:
# Define comeback situations: team trailing by 2 goals
comeback_situations = shots_3p[shots_3p['score_diff'] == -2].copy()
print(f"Shots in comeback situations (down 2): {len(comeback_situations)}")

# Filter out shots after 1110 seconds (goalie-pull regime)
# to avoid confounding from empty-net situations
cutoff_seconds = 1110
comeback_df = comeback_situations[comeback_situations['time'] <= cutoff_seconds].copy()
print(f"After removing shots after {cutoff_seconds}s (empty-net): {len(comeback_df)}")

# Verify no missing xGoal values
print(f"Missing xGoal: {comeback_df['xGoal'].isna().sum()}")

print(f"\nFinal dataset shape: {comeback_df.shape}")
print(f"Games represented: {comeback_df['gameId'].nunique()}")

## 3. Exploratory Data Analysis


### 3.1 Temporal Distribution of Shots


In [ ]:
# Define 5-minute bins
bin_edges = [0, 300, 600, 900, 1110]
bin_labels = ['0-5 min', '5-10 min', '10-15 min', '15-20 min']
comeback_df['time_bin'] = pd.cut(comeback_df['time'], bins=bin_edges, labels=bin_labels, right=False)

# Count shots per bin
shot_counts = comeback_df['time_bin'].value_counts().sort_index()
print("Shot counts by time window:")
print(shot_counts)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
shot_counts.plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)
ax.set_title('Shot Frequency by Time Window (Comeback Situations)', fontsize=12, fontweight='bold')
ax.set_xlabel('Time Window (minutes into 3rd period)')
ax.set_ylabel('Number of Shots')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_shot_counts_by_window.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to {OUTPUT_DIR}")

### 3.2 xGoal Distribution & Summary Statistics


In [ ]:
# Summary statistics
print("xGoal Summary Statistics (All Comeback Shots):")
print(comeback_df['xGoal'].describe())

# By time bin
print("\nxGoal Mean by Time Window:")
mean_xgoal_by_bin = comeback_df.groupby('time_bin')['xGoal'].agg(['count', 'mean', 'std', 'median'])
print(mean_xgoal_by_bin)

# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(comeback_df['xGoal'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('xGoal per Shot')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of xGoal (All Comeback Shots)')
axes[0].axvline(comeback_df['xGoal'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {comeback_df["xGoal"].mean():.3f}')
axes[0].legend()

# Box plot by time bin
comeback_df.boxplot(column='xGoal', by='time_bin', ax=axes[1])
axes[1].set_title('xGoal Distribution by Time Window')
axes[1].set_xlabel('Time Window')
axes[1].set_ylabel('xGoal per Shot')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_xgoal_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.3 Temporal Trajectory (Raw Data)


In [ ]:
# Scatter plot of xGoal vs. time
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(comeback_df['time'], comeback_df['xGoal'], alpha=0.4, s=30, color='steelblue')

# Add binned mean line
bin_midpoints = [150, 450, 750, 1050]
bin_means = mean_xgoal_by_bin['mean'].values
ax.plot(bin_midpoints, bin_means, 'o-', color='red', linewidth=2.5, markersize=8, label='Binned Mean')

ax.set_xlabel('Time Elapsed in 3rd Period (seconds)', fontsize=11)
ax.set_ylabel('xGoal per Shot', fontsize=11)
ax.set_title('Shot Quality Trajectory in Comeback Situations (Down 2, 3rd Period)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_raw_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Visual inspection suggests: potential {'increase' if bin_means[-1] > bin_means[0] else 'decrease'} in xGoal over time")

## 4. Data Validation & Quality Checks


In [ ]:
print("=== DATA QUALITY CHECKLIST ===")
print(f"✓ No missing values in outcome (xGoal): {comeback_df['xGoal'].isna().sum() == 0}")
print(f"✓ No missing values in predictor (time): {comeback_df['time'].isna().sum() == 0}")
print(f"✓ xGoal in expected range [0, ~0.8]: {(comeback_df['xGoal'] >= 0).all() and (comeback_df['xGoal'] <= 1).all()}")
print(f"✓ Time in expected range [0, 1110]: {(comeback_df['time'] >= 0).all() and (comeback_df['time'] <= 1110).all()}")
print(f"✓ All shots from 3rd period: {(comeback_df.get('period', 3) == 3).all()}")
print(f"✓ All shots from trailing situations (score_diff = -2): {(comeback_df['score_diff'] == -2).all()}")

print(f"\nSample size adequacy:")
print(f"  Total shots: {len(comeback_df)}")
print(f"  Unique games: {comeback_df['gameId'].nunique()}")
print(f"  Avg shots per game: {len(comeback_df) / comeback_df['gameId'].nunique():.1f}")

print(f"\n✓ Recommended for modeling: YES (n={len(comeback_df)} is sufficient for GP regression)")

## 5. Export Cleaned Data


In [ ]:
# Save cleaned dataset for next phase
comeback_df_export = comeback_df[['gameId', 'time', 'xGoal', 'score_diff', 'time_bin']].copy()
comeback_df_export.to_csv(OUTPUT_DIR / 'comeback_clean.csv', index=False)

print(f"Cleaned dataset exported to {OUTPUT_DIR / 'comeback_clean.csv'}")
print(f"Shape: {comeback_df_export.shape}")
print(f"\nReady for Phase 2: Experimentation (GP regression modeling)")